# FloorPlan3D: walls, doors and windows from floor plan images

This notebook trains the model behind FloorPlan3D's **Image to plan**: a U-Net that labels every pixel of a floor plan image as
*background*, *wall*, *door* or *window*. It runs top to bottom on Kaggle's two T4 GPUs and leaves the trained model, its scores
on the 400 test plans and an ONNX copy for the browser in its output.

| Section | What it shows |
|---|---|
| 0. Settings | the run's settings and the machine |
| 1. The data | CubiCasa5k: 5,000 plans in three kinds, split 4,200 / 400 / 400 |
| 2. Labels | the hand-traced polygons drawn as one class map per plan, exactly as the app's scorer draws them |
| 3. Training crops | 512 px crops with scale, rotation and scan-like noise |
| 4. The model | a plain U-Net, 7.8 M parameters |
| 5. The loss | cross-entropy plus soft Dice |
| 6. Training | one process per GPU, half precision, a time-budgeted schedule |
| 7. How training went | loss and validation scores over time |
| 8. The test plans | wall IoU on all 400 test plans, against the hand-written rules of the app |
| 9. Where it goes wrong | the best, a typical and the worst test plan, pixel by pixel |
| 10. Export for the browser | the model as ONNX, checked against PyTorch |

Every code cell below is cut from the FloorPlan3D repository's `training/` modules by `training/kaggle/build_notebook.py`, so this
notebook runs exactly the code its tests check.

**Data and licence.** CubiCasa5k (Kalervo et al., *CubiCasa5K: A Dataset and an Improved Multi-Task Model for Floorplan Image
Analysis*, SCIA 2019) is licensed CC BY-NC-SA 4.0. It, this notebook's outputs and the model trained here are for non-commercial
use only.

## 0. Settings

`RUN` holds everything that shapes the run. On Kaggle the defaults train for the time budget in `minutes`; the learning-rate
schedule stretches to fit it. Set the environment variable `FP3D_QUICK=1` to check the notebook end to end in a couple of minutes
on any machine, with a tiny model and a handful of plans.

In [ ]:
import os

ON_KAGGLE = os.path.isdir('/kaggle/input')
QUICK = os.environ.get('FP3D_QUICK') == '1'

RUN = dict(
    width=32,          # U-Net width: 32 channels at full resolution, 7.8 M parameters
    crop=512,          # training crops are 512 × 512 px
    batch=8,           # crops per GPU per step
    per_plan=4,        # crops cut from each plan that is loaded
    minutes=180,       # training time budget; the schedule fits itself to it
    steps=60000,       # step budget; whichever runs out first ends training
    lr=2e-3,           # peak learning rate (AdamW), after a 300-step warm-up, then cosine down to 2%
    val_plans=60,      # validation plans scored every val_every steps; the best checkpoint is kept
    val_every=3000,
    test_limit=400,   # test plans scored at the end
    train_limit=0,     # 0 = all 4,200 training plans
    workers=2,         # data-loading processes per GPU
    log_every=200,
)
if QUICK:
    RUN.update(width=8, crop=128, minutes=1, steps=30, val_plans=2, val_every=15, test_limit=3, train_limit=12, workers=0, log_every=10, device='cpu')
RUN

In [ ]:
from __future__ import annotations

import json
import math
import re
import struct
import subprocess
import sys
import time
import warnings
from dataclasses import MISSING, asdict, dataclass, fields
from multiprocessing import Pool

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch.utils.data import DataLoader, DistributedSampler, RandomSampler

if ON_KAGGLE:
    DATA = next(d for d, _, files in os.walk('/kaggle/input') if 'train.txt' in files)
    LABELS = '/tmp/labels'          # drawn class maps (not part of the output)
    OUT = '/kaggle/working'         # everything kept: model, scores, history, ONNX
else:
    DATA = os.environ.get('FP3D_DATA', 'data/cubicasa5k/cubicasa5k')
    LABELS = os.environ.get('FP3D_LABELS', 'data/cubicasa5k-labels')
    OUT = os.environ.get('FP3D_OUT', 'data/models/notebook-run')
os.makedirs(OUT, exist_ok=True)


def gpu_names() -> list[str]:
    """The GPUs, from nvidia-smi. CUDA itself is not touched in this process before training: the training processes are forked
    from it, and a forked process cannot use a CUDA context its parent has already opened."""
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, timeout=30)
        return [line.strip() for line in out.stdout.splitlines() if line.strip()]
    except (OSError, subprocess.SubprocessError):
        return []


GPUS = gpu_names()
WORLD = len(GPUS) or int(os.environ.get('FP3D_WORLD', 1))  # one training process per GPU
print(f'Python {sys.version.split()[0]}, PyTorch {torch.__version__}, OpenCV {cv2.__version__}')
print(f'GPUs: {GPUS or "none"}; {os.cpu_count()} CPU cores; {WORLD} training process(es)')
print(f'data {DATA}\nlabels {LABELS}\noutput {OUT}')

## 1. The data

CubiCasa5k holds 5,000 Finnish floor plans, each a raster image (`F1_scaled.png`) with every wall, door, window and room traced
by hand as polygons in `model.svg`. They come in three kinds: *high-quality architectural* drawings (thin, outlined walls, hatching,
dense labels), *high-quality* and *colorful* (solid, often coloured walls). The dataset fixes the split: 4,200 plans to train on,
400 to choose the checkpoint with (validation) and 400 to report on (test), which are used once, at the end.

In [ ]:
BACKGROUND, WALL, DOOR, WINDOW = 0, 1, 2, 3


CLASSES = ('background', 'wall', 'door', 'window')


def read_split(root: str, split: str) -> list[str]:
    """Plan folders (relative to root, like 'high_quality_architectural/1191') of a split list."""
    with open(os.path.join(root, f'{split}.txt')) as f:
        return [line.strip().strip('/') for line in f if line.strip()]


def plan_name(plan: str) -> str:
    return plan.replace('/', '_')


def png_size(path: str) -> tuple[int, int]:
    """(width, height) from a PNG's header, without decoding the image."""
    with open(path, 'rb') as f:
        head = f.read(24)
    if head[:8] != b'\x89PNG\r\n\x1a\n':
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        return img.shape[1], img.shape[0]
    return struct.unpack('>II', head[16:24])

In [ ]:
splits = {split: read_split(DATA, split) for split in ('train', 'val', 'test')}
kind = lambda plan: plan.split('/')[0]
counts = pd.DataFrame({split: pd.Series([kind(p) for p in plans]).value_counts() for split, plans in splits.items()}).fillna(0).astype(int)
counts.loc['total'] = counts.sum()
display(counts)

sizes = np.array([png_size(os.path.join(DATA, p, 'F1_scaled.png')) for plans in splits.values() for p in plans])
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(sizes.max(1), bins=50, color='#2f70bb')
ax.set_xlabel('longest side of the plan image (px)')
ax.set_ylabel('plans')
ax.spines[['top', 'right']].set_visible(False)
plt.show()
print(f'median size {int(np.median(sizes[:, 0]))} × {int(np.median(sizes[:, 1]))} px; largest side up to {sizes.max()} px')

## 2. Labels: from polygons to pixels

Each plan's walls, doors and windows are read from `model.svg` and drawn as one **class map** on the pixel grid of its image:
0 background, 1 wall, 2 door, 3 window. Doors and windows are drawn over the walls, so *wall* means wall with its openings cut out,
the same truth the app's TypeScript scorer (`scripts/evaluate-cubicasa.ts`) uses for the rules.

Two details matter for a fair comparison. Only the first polygon directly inside each `Wall`, `Door` or `Window` group is the shape
(the others are decorations), and polygons are filled by scanlines through pixel centres with the even–odd rule. `fill_polygon` is a
vectorised port of the app's `fillPolygon`; on all 5,000 plans the two draw identical maps (0 pixels differ).

In [ ]:
_TAG = re.compile(r'<(/?)(g|polygon)\b([^>]*?)(/?)>')


def _attribute(name: str, text: str) -> str | None:
    m = re.search(rf'\b{name}\s*=\s*"([^"]*)"', text)
    return m.group(1) if m else None


def _points(text: str) -> np.ndarray:
    pts = []
    for pair in text.split():
        xy = pair.split(',')
        if len(xy) != 2:
            continue
        try:
            x, y = float(xy[0]), float(xy[1])
        except ValueError:
            continue
        if np.isfinite(x) and np.isfinite(y):
            pts.append((x, y))
    return np.array(pts, dtype=np.float64).reshape(-1, 2)


def parse_svg(svg: str) -> dict[str, list[np.ndarray]]:
    """Wall, Door and Window polygons of a CubiCasa5k model.svg (see cubicasaSvg.ts)."""
    found: dict[str, list[np.ndarray]] = {'Wall': [], 'Door': [], 'Window': []}
    stack: list[list] = []  # [id, first polygon taken]
    for m in _TAG.finditer(svg):
        closing, tag, attributes, self_closing = m.groups()
        if tag == 'g':
            if closing:
                if stack:
                    stack.pop()
            elif not self_closing:
                stack.append([_attribute('id', attributes) or '', False])
            continue
        if closing or not stack or stack[-1][1]:
            continue
        parent = stack[-1]
        if parent[0] not in found:
            continue
        pts = _points(_attribute('points', attributes) or '')
        if len(pts) >= 3:
            found[parent[0]].append(pts)
        parent[1] = True
    return found


def fill_polygon(mask: np.ndarray, pts: np.ndarray, value: int) -> None:
    """Fills a polygon exactly as fillPolygon in src/vision/maskScore.ts does: scanlines through pixel
    centres, even-odd rule, a pixel filled when its centre lies between a pair of crossings."""
    h, w = mask.shape
    if len(pts) < 3:
        return
    y0, y1 = max(0, int(np.floor(pts[:, 1].min()))), min(h - 1, int(np.ceil(pts[:, 1].max())))
    bx0, bx1 = max(0, int(np.floor(pts[:, 0].min()))), min(w - 1, int(np.ceil(pts[:, 0].max())))
    if y1 < y0 or bx1 < bx0:
        return
    cy = np.arange(y0, y1 + 1, dtype=np.float64)[:, None] + 0.5
    xi, yi = pts[:, 0][None], pts[:, 1][None]
    xj, yj = np.roll(pts[:, 0], 1)[None], np.roll(pts[:, 1], 1)[None]  # the previous point, as in the TS loop
    crosses = (yi > cy) != (yj > cy)
    with np.errstate(divide='ignore', invalid='ignore'):
        xc = np.where(crosses, xi + (cy - yi) * (xj - xi) / (yj - yi), np.inf)
    xc.sort(axis=1)
    pairs = xc.shape[1] // 2
    a, b = xc[:, 0:2 * pairs:2], xc[:, 1:2 * pairs:2]
    ok = np.isfinite(a) & np.isfinite(b)
    xa = np.clip(np.ceil(np.where(ok, a, 0) - 0.5), 0, w).astype(np.int64)
    xb = np.clip(np.floor(np.where(ok, b, 0) - 0.5), -1, w - 1).astype(np.int64)
    ok &= xb >= xa
    if not ok.any():
        return
    rows = np.broadcast_to(np.arange(cy.shape[0])[:, None], a.shape)[ok]
    span = bx1 - bx0 + 1
    diff = np.zeros((cy.shape[0], span + 1), np.int32)
    np.add.at(diff, (rows, xa[ok] - bx0), 1)
    np.add.at(diff, (rows, xb[ok] - bx0 + 1), -1)
    inside = np.cumsum(diff, axis=1)[:, :span] > 0
    mask[y0:y1 + 1, bx0:bx1 + 1][inside] = value


def label_map(svg: str, width: int, height: int) -> np.ndarray:
    """The class map of one plan, on the pixel grid of its F1_scaled.png."""
    shapes = parse_svg(svg)
    labels = np.zeros((height, width), np.uint8)
    for name, value in (('Wall', WALL), ('Door', DOOR), ('Window', WINDOW)):
        for pts in shapes[name]:
            fill_polygon(labels, pts, value)
    return labels


def _prepare_one(job: tuple[str, str, str]) -> int:
    root, plan, out_dir = job
    target = os.path.join(out_dir, plan_name(plan) + '.png')
    if os.path.exists(target):
        return 0
    folder = os.path.join(root, plan)
    width, height = png_size(os.path.join(folder, 'F1_scaled.png'))
    with open(os.path.join(folder, 'model.svg'), encoding='utf-8') as f:
        labels = label_map(f.read(), width, height)
    cv2.imwrite(target, labels)
    return 1


def prepare_labels(root: str, out_dir: str, splits=('train', 'val', 'test'), workers: int | None = None) -> int:
    """Draws every plan's class map once into out_dir/<plan_name>.png. Returns how many were made."""
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(root, plan, out_dir) for split in splits for plan in read_split(root, split)]
    jobs = [job for job in jobs if not os.path.exists(os.path.join(out_dir, plan_name(job[1]) + '.png'))]
    if not jobs:
        return 0
    with Pool(workers or os.cpu_count()) as pool:
        return sum(pool.imap_unordered(_prepare_one, jobs, chunksize=16))

In [ ]:
t = time.time()
made = prepare_labels(DATA, LABELS)
open(os.path.join(LABELS, '.done'), 'w').close()
print(f'{made} class maps drawn in {time.time() - t:.0f} s ({len(os.listdir(LABELS)) - 1} in {LABELS})')

In [ ]:
# Background, wall, door, window: white, teal, amber, magenta.
COLOURS = np.array([[255, 255, 255], [27, 108, 128], [201, 133, 0], [213, 81, 129]], np.uint8)


def load(plan: str) -> tuple[np.ndarray, np.ndarray]:
    gray = cv2.imread(os.path.join(DATA, plan, 'F1_scaled.png'), cv2.IMREAD_GRAYSCALE)
    labels = cv2.imread(os.path.join(LABELS, plan_name(plan) + '.png'), cv2.IMREAD_UNCHANGED)
    return gray, labels


def overlay(gray: np.ndarray, labels: np.ndarray, alpha: float = 0.75) -> np.ndarray:
    """The plan in light grey with each labelled class painted over it."""
    base = np.repeat((155 + gray[..., None] * 0.39).astype(np.uint8), 3, axis=2)
    painted = COLOURS[labels]
    out = base.copy()
    mask = labels > 0
    out[mask] = (alpha * painted[mask] + (1 - alpha) * base[mask]).astype(np.uint8)
    return out


def around(mask: np.ndarray, *images: np.ndarray, margin: float = 0.06) -> list[np.ndarray]:
    """The images cut to where the mask is set, plus a margin: many plans are a small drawing on a large sheet."""
    ys, xs = np.nonzero(mask)
    if ys.size == 0:
        return list(images)
    h, w = mask.shape
    my, mx = int(margin * (ys.max() - ys.min())) + 10, int(margin * (xs.max() - xs.min())) + 10
    y0, y1, x0, x1 = max(0, ys.min() - my), min(h, ys.max() + my), max(0, xs.min() - mx), min(w, xs.max() + mx)
    return [image[y0:y1, x0:x1] for image in images]


examples = [next(p for p in splits['train'] if kind(p) == k) for k in ('high_quality_architectural', 'high_quality', 'colorful')]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for col, plan in enumerate(examples):
    gray, labels = around(load(plan)[1] > 0, *load(plan))
    axes[0, col].imshow(gray, cmap='gray')
    axes[0, col].set_title(plan)
    axes[1, col].imshow(overlay(gray, labels))
    axes[1, col].set_title('wall (teal), door (amber), window (magenta)')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# How much of a plan each class covers, over 300 training plans.
pixels = np.zeros(4)
for plan in splits['train'][:300]:
    pixels += np.bincount(load(plan)[1].ravel(), minlength=4)
shares = pd.DataFrame({'class': CLASSES, 'share of pixels': (pixels / pixels.sum()).round(4)}).set_index('class')
display(shares)

Walls cover only a few percent of a plan and doors and windows well under one percent. A loss that scores every pixel the same can
do well by saying "background" almost everywhere, which is why the loss in section 5 adds a Dice term per class.

## 3. Training crops

Plans are large and their sizes vary, so the model trains on 512 × 512 px crops. Each plan that is loaded gives four crops, most of
them centred on a wall, door or window pixel so they are not mostly empty paper. Every crop is scaled by 0.7–1.4×, turned by a
multiple of 90°, maybe mirrored, then made to look like another scan: contrast, faded or heavy ink, blur, grain and a grey
background. The input is *ink*: 0 for paper and 1 for black, so the zero padding at the edges reads as paper.

In [ ]:
def ink(gray: np.ndarray) -> np.ndarray:
    """Model input: 0 for white paper, 1 for black ink (so zero padding reads as paper)."""
    return 1.0 - gray.astype(np.float32) / 255.0


class TrainCrops:
    """Random training crops from whole plans: a few per plan load, with scale, rotation and scan-noise augmentation.

    Used as a torch Dataset (it has __len__ and __getitem__); returns (x [K,1,S,S] float32, y [K,S,S] int64).
    """

    def __init__(self, root: str, labels_dir: str, plans: list[str], size: int = 512, per_plan: int = 4, augment: bool = True):
        self.items = [(os.path.join(root, p, 'F1_scaled.png'), os.path.join(labels_dir, plan_name(p) + '.png')) for p in plans]
        self.size, self.per_plan, self.augment = size, per_plan, augment

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i: int):
        image_path, label_path = self.items[i]
        gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        labels = cv2.imread(label_path, cv2.IMREAD_UNCHANGED)
        rng = np.random.default_rng()
        xs, ys = zip(*(self._crop(gray, labels, rng) for _ in range(self.per_plan)))
        return np.stack(xs)[:, None], np.stack(ys).astype(np.int64)

    def _crop(self, gray: np.ndarray, labels: np.ndarray, rng: np.random.Generator):
        h, w = gray.shape
        s = self.size
        scale = float(np.exp(rng.uniform(np.log(0.7), np.log(1.4)))) if self.augment else 1.0
        r = max(32, int(round(s / scale)))  # side of the region taken from the plan

        # Centre: most of the time on a wall, door or window pixel, so crops are not mostly paper.
        cy, cx = rng.integers(0, h), rng.integers(0, w)
        if rng.random() < 0.7:
            py, px = rng.integers(0, h, 96), rng.integers(0, w, 96)
            hit = np.flatnonzero(labels[py, px] > 0)
            if hit.size:
                cy, cx = py[hit[0]], px[hit[0]]
        y0 = int(cy - r // 2 + rng.integers(-r // 4, r // 4 + 1))
        x0 = int(cx - r // 2 + rng.integers(-r // 4, r // 4 + 1))
        y0 = min(max(y0, 0), max(h - r, 0)) if h >= r else -int(rng.integers(0, r - h + 1))
        x0 = min(max(x0, 0), max(w - r, 0)) if w >= r else -int(rng.integers(0, r - w + 1))

        img = np.full((r, r), 255, np.uint8)
        lab = np.zeros((r, r), np.uint8)
        sy0, sx0 = max(y0, 0), max(x0, 0)
        sy1, sx1 = min(y0 + r, h), min(x0 + r, w)
        img[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = gray[sy0:sy1, sx0:sx1]
        lab[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = labels[sy0:sy1, sx0:sx1]
        if r != s:
            img = cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA if r > s else cv2.INTER_LINEAR)
            lab = cv2.resize(lab, (s, s), interpolation=cv2.INTER_NEAREST)

        if self.augment:
            k = int(rng.integers(0, 4))
            img, lab = np.rot90(img, k), np.rot90(lab, k)
            if rng.random() < 0.5:
                img, lab = img[:, ::-1], lab[:, ::-1]
            x = ink(np.ascontiguousarray(img))
            # Scan-like variation: contrast, faded or heavy ink, blur, grain, a grey background.
            x = x * rng.uniform(0.6, 1.15) + rng.uniform(-0.05, 0.12) * (rng.random() < 0.3)
            if rng.random() < 0.25:
                x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.4, 1.2))
            if rng.random() < 0.3:
                x = x + rng.normal(0, rng.uniform(0.01, 0.06), x.shape).astype(np.float32)
            x = np.clip(x, 0, 1).astype(np.float32)
            return x, np.ascontiguousarray(lab)
        return ink(img), lab

In [ ]:
preview = TrainCrops(DATA, LABELS, splits['train'][:200], size=RUN['crop'], per_plan=4)
crops = [preview[i] for i in (0, 1)]
xs = np.concatenate([c[0] for c in crops])[:, 0]
ys = np.concatenate([c[1] for c in crops])
fig, axes = plt.subplots(2, 8, figsize=(20, 5.4))
for i in range(8):
    axes[0, i].imshow(1 - xs[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].imshow(COLOURS[ys[i]])
for ax in axes.ravel():
    ax.axis('off')
axes[0, 0].set_title('input (ink shown dark)', loc='left')
axes[1, 0].set_title('target classes', loc='left')
plt.tight_layout()
plt.show()

## 4. The model

A plain U-Net (Ronneberger et al., 2015): five levels of two 3 × 3 convolutions with batch norm, max-pooling on the way down,
transposed convolutions and skip connections on the way up, and a 1 × 1 convolution to four class scores per pixel. At width 32 the
levels have 32, 64, 128, 256 and 512 channels: 7.8 M parameters.

Whole plans are predicted in overlapping 1,024 px tiles (`predict_plan`), keeping only the middle of each tile, so a plan of any
size fits in memory and tile edges never show.

In [ ]:
def _block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, width: int = 32, levels: int = 5, classes: int = 4):
        super().__init__()
        widths = [width * 2 ** i for i in range(levels)]
        self.config = dict(width=width, levels=levels, classes=classes)
        self.down = nn.ModuleList()
        cin = 1
        for w in widths:
            self.down.append(_block(cin, w))
            cin = w
        self.up = nn.ModuleList()
        self.fuse = nn.ModuleList()
        for w in reversed(widths[:-1]):
            self.up.append(nn.ConvTranspose2d(cin, w, 2, stride=2))
            self.fuse.append(_block(2 * w, w))
            cin = w
        self.head = nn.Conv2d(cin, classes, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for i, down in enumerate(self.down):
            x = down(x)
            if i < len(self.down) - 1:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for up, fuse in zip(self.up, self.fuse):
            x = fuse(torch.cat([up(x), skips.pop()], 1))
        return self.head(x)


@torch.no_grad()
def predict_plan(model: nn.Module, x: np.ndarray, device: torch.device, tile: int = 1024, margin: int = 96) -> np.ndarray:
    """Class map of a whole plan (x = ink, H×W float32), predicted in overlapping tiles so any size fits in memory."""
    h, w = x.shape
    H = math.ceil(max(h, tile) / 32) * 32
    W = math.ceil(max(w, tile) / 32) * 32
    canvas = np.zeros((H, W), np.float32)
    canvas[:h, :w] = x
    out = np.zeros((H, W), np.uint8)
    step = tile - 2 * margin

    def starts(n: int) -> list[int]:
        s = list(range(0, n - tile + 1, step))
        if s[-1] != n - tile:
            s.append(n - tile)
        return s

    amp = device.type == 'cuda'
    for y0 in starts(H):
        for x0 in starts(W):
            t = torch.from_numpy(canvas[y0:y0 + tile, x0:x0 + tile])[None, None].to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                pred = model(t).argmax(1)[0].to('cpu', torch.uint8).numpy()
            top = 0 if y0 == 0 else margin
            left = 0 if x0 == 0 else margin
            bottom = tile if y0 + tile >= H else tile - margin
            right = tile if x0 + tile >= W else tile - margin
            out[y0 + top:y0 + bottom, x0 + left:x0 + right] = pred[top:bottom, left:right]
    return out[:h, :w]

In [ ]:
# Built here only to count its parameters. Running it waits for the training processes: they are forked from this notebook,
# and a forked process should not inherit a CPU thread pool (or a GPU context) that PyTorch has already started.
net = UNet(RUN['width'])
s = RUN['crop']
display(pd.DataFrame({'channels': [RUN['width'] * 2 ** i for i in range(5)], 'resolution of a crop': [f'{s >> i} × {s >> i}' for i in range(5)]},
                     index=pd.Index(range(1, 6), name='level')))
print(f'{sum(p.numel() for p in net.parameters()) / 1e6:.2f} M parameters; output: 4 class scores for every input pixel')
del net

## 5. The loss

Cross-entropy scores every pixel; soft Dice scores the overlap of each class as a whole, so thin walls and small doors count as much
as they should. The Dice term covers wall, door and window; background is left out because it is most of every plan.

$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log p_{i,\,y_i} \;+\; \frac{1}{3}\sum_{c\,\in\,\{\text{wall},\,\text{door},\,\text{window}\}}
\left(1 - \frac{2\sum_i p_{i,c}\,g_{i,c} + 1}{\sum_i p_{i,c} + \sum_i g_{i,c} + 1}\right)
$$

$p_{i,c}$ is the predicted probability that pixel $i$ is class $c$, $y_i$ its true class and $g_{i,c}$ 1 when that class is $c$.
Scores are reported as IoU: $|F \cap T| \,/\, |F \cup T|$ for the pixels found $F$ and the true pixels $T$ of a class.

In [ ]:
def collate(batch):
    xs, ys = zip(*batch)
    return torch.from_numpy(np.concatenate(xs)), torch.from_numpy(np.concatenate(ys))


def dice_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Soft Dice over wall, door and window (background left out: it is most of every plan)."""
    probs = logits.softmax(1)
    onehot = F.one_hot(target, probs.shape[1]).permute(0, 3, 1, 2).to(probs.dtype)
    inter = (probs * onehot).sum((0, 2, 3))
    denom = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return (1 - (2 * inter + 1) / (denom + 1))[1:].mean()


def iou(a: np.ndarray, b: np.ndarray) -> float | None:
    union = np.count_nonzero(a | b)
    return None if union == 0 else np.count_nonzero(a & b) / union


def plan_scores(pred: np.ndarray, labels: np.ndarray) -> dict:
    found, truth = pred == WALL, labels == WALL
    overlap = np.count_nonzero(found & truth)
    nf, nt = np.count_nonzero(found), np.count_nonzero(truth)
    openings_pred = (pred == DOOR) | (pred == WINDOW)
    openings_true = (labels == DOOR) | (labels == WINDOW)
    return {
        'iou': iou(found, truth) or 0.0,
        'precision': overlap / nf if nf else 0.0,
        'recall': overlap / nt if nt else 0.0,
        'openings_iou': iou(openings_pred, openings_true),
    }

## 6. Training

`train_worker` runs once per GPU. With two T4s the two processes train one model together (PyTorch DistributedDataParallel): each
takes its own crops, and their gradients are averaged every step. They are forked from this notebook, so all the code above is
theirs too. Settings that make the T4s fast: half precision with a gradient scaler, channels-last memory, and cuDNN autotuning.

The learning rate warms up over 300 steps and then follows a cosine down to 2% of its peak. Its clock is whichever budget is
further along, steps or minutes, so a time-limited run still ends with a full cool-down. Every `val_every` steps the model is scored
on the validation plans and the best checkpoint is kept. At the end that checkpoint predicts every test plan: its scores go to
`test_results.json`, and its class maps to `pred_test/`, where the app's own scorer can read them too.

In [ ]:
@dataclass
class TrainConfig:
    data: str  # folder holding train.txt, val.txt, test.txt
    labels: str  # folder of class-map PNGs (drawn if missing)
    out: str
    width: int = 32  # U-Net width: 32 is 7.8 M parameters
    crop: int = 512
    batch: int = 8  # crops per GPU per step
    per_plan: int = 4  # crops taken from each plan load
    steps: int = 24000
    minutes: float = 95  # training time budget; the schedule ends at whichever comes first
    lr: float = 2e-3
    val_plans: int = 60
    val_every: int = 3000
    test_limit: int = 400
    train_limit: int = 0  # use only the first N training plans (0 = all; for quick checks)
    workers: int = 2  # data loader processes per GPU
    log_every: int = 200
    device: str = 'auto'


def train_worker(rank: int, world: int, cfg: TrainConfig) -> None:
    """Trains on one GPU (or the CPU) as process `rank` of `world`; with world > 1 the processes train one model together (DDP)."""
    distributed = world > 1
    if distributed:
        os.environ.setdefault('MASTER_ADDR', '127.0.0.1')
        os.environ.setdefault('MASTER_PORT', '29531')
        if torch.cuda.is_available():
            torch.cuda.set_device(rank % torch.cuda.device_count())
            device = torch.device('cuda', torch.cuda.current_device())
            dist.init_process_group('nccl', rank=rank, world_size=world, device_id=device)
        else:
            device = torch.device('cpu')
            dist.init_process_group('gloo', rank=rank, world_size=world)
        # A process forked from one that already used PyTorch's or OpenCV's CPU thread pools (the notebook) cannot safely use the
        # pools it inherited. One thread is plenty: the GPUs do the work, and the data loaders are separate processes.
        torch.set_num_threads(1)
        cv2.setNumThreads(0)  # the same for OpenCV's thread pool; the data loaders inherit this
    else:
        name = cfg.device if cfg.device != 'auto' else ('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
        device = torch.device(name)
    cuda = device.type == 'cuda'
    if cuda:
        torch.backends.cudnn.benchmark = True
    os.makedirs(cfg.out, exist_ok=True)
    log_file = open(os.path.join(cfg.out, 'log.txt'), 'a') if rank == 0 else None
    history: dict = {'config': asdict(cfg), 'world': world, 'device': device.type, 'loss': [], 'val': []}

    def log(msg: str) -> None:
        if rank == 0:
            line = f'[{time.strftime("%H:%M:%S")}] {msg}'
            print(line, flush=True)
            log_file.write(line + '\n')
            log_file.flush()

    def record(kind: str, entry: dict) -> None:
        if rank == 0:
            history[kind].append(entry)
            with open(os.path.join(cfg.out, 'history.json'), 'w') as f:
                json.dump(history, f, indent=1)

    def barrier() -> None:
        if distributed:
            dist.barrier()

    if rank == 0 and not os.path.exists(os.path.join(cfg.labels, '.done')):
        t = time.time()
        made = prepare_labels(cfg.data, cfg.labels)
        open(os.path.join(cfg.labels, '.done'), 'w').close()
        log(f'labels drawn for {made} plans in {time.time() - t:.0f} s')
    barrier()

    train_plans = read_split(cfg.data, 'train')
    if cfg.train_limit:
        train_plans = train_plans[:cfg.train_limit]
    val_plans = read_split(cfg.data, 'val')[:cfg.val_plans]
    test_plans = read_split(cfg.data, 'test')[:cfg.test_limit]
    dataset = TrainCrops(cfg.data, cfg.labels, train_plans, size=cfg.crop, per_plan=cfg.per_plan)
    sampler = DistributedSampler(dataset, shuffle=True) if distributed else RandomSampler(dataset)
    loader = DataLoader(dataset, batch_size=max(1, cfg.batch // cfg.per_plan), sampler=sampler, num_workers=cfg.workers,
                        collate_fn=collate, pin_memory=cuda, drop_last=True, persistent_workers=cfg.workers > 0,
                        prefetch_factor=4 if cfg.workers > 0 else None)

    net = UNet(cfg.width).to(device)
    if cuda:
        net = net.to(memory_format=torch.channels_last)
    model = (
        # Gradients live inside DDP's communication buckets, so their memory layout always matches (and is not copied).
        torch.nn.parallel.DistributedDataParallel(net, device_ids=[device.index] if cuda else None, gradient_as_bucket_view=True)
        if distributed else net
    )
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=cuda)
    params = sum(p.numel() for p in net.parameters()) / 1e6
    log(f'{world} × {device.type}, {len(train_plans)} train plans, {params:.1f} M params, '
        f'{cfg.batch * world} crops per step, up to {cfg.steps} steps or {cfg.minutes:.0f} min')

    def evaluate(plans: list[str], pred_dir: str | None = None) -> tuple[dict, list[dict]]:
        net.eval()
        rows = []
        for i, plan in enumerate(plans):
            if i % world != rank:
                continue
            gray = cv2.imread(os.path.join(cfg.data, plan, 'F1_scaled.png'), cv2.IMREAD_GRAYSCALE)
            labels = cv2.imread(os.path.join(cfg.labels, plan_name(plan) + '.png'), cv2.IMREAD_UNCHANGED)
            t = time.time()
            pred = predict_plan(net, ink(gray), device)
            ms = (time.time() - t) * 1000
            if pred_dir:
                cv2.imwrite(os.path.join(pred_dir, plan_name(plan) + '.png'), pred)
            rows.append({'plan': plan, 'width': gray.shape[1], 'height': gray.shape[0], 'ms': ms, **plan_scores(pred, labels)})
        net.train()
        sums = torch.tensor([sum(r['iou'] for r in rows), sum(r['precision'] for r in rows), sum(r['recall'] for r in rows),
                             sum(r['openings_iou'] for r in rows if r['openings_iou'] is not None),
                             sum(1 for r in rows if r['openings_iou'] is not None), len(rows)], dtype=torch.float64,
                            device=device if distributed and cuda else 'cpu')
        if distributed:
            dist.all_reduce(sums)
        s = sums.tolist()
        n = max(s[5], 1)
        return {'iou': s[0] / n, 'precision': s[1] / n, 'recall': s[2] / n, 'openings_iou': s[3] / max(s[4], 1), 'plans': int(s[5])}, rows

    def save(path: str, step: int, metrics: dict) -> None:
        if rank == 0:
            torch.save({'state': net.state_dict(), 'config': net.config, 'step': step, 'val': metrics}, path)

    start = time.time()
    budget = cfg.minutes * 60
    step, time_frac, best = 0, 0.0, -1.0
    seen = 0
    sample_every = max(1, cfg.log_every // 4)  # the loss is read (a GPU sync) only this often
    window_start, window_loss, window_samples = time.time(), 0.0, 0
    epoch = 0
    model.train()
    done = False
    while not done:
        if distributed:
            sampler.set_epoch(epoch)
        for x, y in loader:
            progress = max(step / cfg.steps, time_frac)
            if progress >= 1:
                done = True
                break
            warm = min(1.0, (step + 1) / 300)
            lr = cfg.lr * warm * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progress)))
            for g in opt.param_groups:
                g['lr'] = lr
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if cuda:
                x = x.contiguous(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                logits = model(x)
            logits = logits.float()
            loss = F.cross_entropy(logits, y) + dice_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            step += 1
            seen += x.shape[0] * world
            if step % sample_every == 0:
                window_loss += loss.item()
                window_samples += 1

            if step % 50 == 0:
                frac = torch.tensor([(time.time() - start) / budget], device=device)
                if distributed:
                    dist.all_reduce(frac, op=dist.ReduceOp.MAX)
                time_frac = float(frac.item())
            if step % cfg.log_every == 0:
                rate = cfg.log_every * x.shape[0] * world / (time.time() - window_start)
                minutes = (time.time() - start) / 60
                mean_loss = window_loss / max(window_samples, 1)
                log(f'step {step:6d}  loss {mean_loss:.4f}  lr {lr:.2e}  {rate:5.1f} crops/s  {minutes:5.1f} min  {seen / 1000:.0f}k crops')
                record('loss', {'step': step, 'loss': mean_loss, 'lr': lr, 'crops_per_s': rate, 'minutes': minutes})
                window_start, window_loss, window_samples = time.time(), 0.0, 0
            if step % cfg.val_every == 0:
                metrics, _ = evaluate(val_plans)
                log(f'val on {metrics["plans"]} plans: wall IoU {metrics["iou"]:.3f} (precision {metrics["precision"]:.3f}, '
                    f'recall {metrics["recall"]:.3f}), door+window IoU {metrics["openings_iou"]:.3f}')
                record('val', {'step': step, 'minutes': (time.time() - start) / 60, **metrics})
                if metrics['iou'] > best:
                    best = metrics['iou']
                    save(os.path.join(cfg.out, 'model_best.pt'), step, metrics)
                window_start = time.time()
        epoch += 1

    metrics, _ = evaluate(val_plans)
    log(f'final val on {metrics["plans"]} plans: wall IoU {metrics["iou"]:.3f} (precision {metrics["precision"]:.3f}, '
        f'recall {metrics["recall"]:.3f}), door+window IoU {metrics["openings_iou"]:.3f}  after {step} steps, {seen / 1000:.0f}k crops, '
        f'{(time.time() - start) / 60:.1f} min')
    record('val', {'step': step, 'minutes': (time.time() - start) / 60, 'final': True, **metrics})
    save(os.path.join(cfg.out, 'model_last.pt'), step, metrics)
    if metrics['iou'] > best:
        best = metrics['iou']
        save(os.path.join(cfg.out, 'model_best.pt'), step, metrics)
    barrier()

    if test_plans:
        state = torch.load(os.path.join(cfg.out, 'model_best.pt'), map_location=device)
        net.load_state_dict(state['state'])
        pred_dir = os.path.join(cfg.out, 'pred_test')
        os.makedirs(pred_dir, exist_ok=True)
        summary, rows = evaluate(test_plans, pred_dir)
        with open(os.path.join(cfg.out, f'test_rows_{rank}.json'), 'w') as f:
            json.dump(rows, f)
        barrier()
        if rank == 0:
            all_rows = []
            for r in range(world):
                path = os.path.join(cfg.out, f'test_rows_{r}.json')
                all_rows += json.load(open(path))
                os.remove(path)
            all_rows.sort(key=lambda r: r['plan'])
            summary['median_iou'] = float(np.median([r['iou'] for r in all_rows]))
            summary['ms_per_plan'] = float(np.mean([r['ms'] for r in all_rows]))
            summary['best_step'] = state['step']
            json.dump({'summary': summary, 'rows': all_rows}, open(os.path.join(cfg.out, 'test_results.json'), 'w'), indent=1)
            log(f'test on {summary["plans"]} plans (checkpoint of step {state["step"]}): wall IoU {summary["iou"]:.3f} '
                f'(median {summary["median_iou"]:.3f}, precision {summary["precision"]:.3f}, recall {summary["recall"]:.3f}), '
                f'door+window IoU {summary["openings_iou"]:.3f}, {summary["ms_per_plan"]:.0f} ms per plan')
    barrier()
    if log_file:
        log_file.close()
    if distributed:
        dist.destroy_process_group()

In [ ]:
cfg = TrainConfig(data=DATA, labels=LABELS, out=OUT, **RUN)
for name in ('log.txt', 'history.json'):  # a fresh history for this run
    if os.path.exists(os.path.join(OUT, name)):
        os.remove(os.path.join(OUT, name))
started = time.time()
if WORLD > 1:
    import torch.multiprocessing as mp
    mp.start_processes(train_worker, args=(WORLD, cfg), nprocs=WORLD, start_method='fork')
else:
    train_worker(0, 1, cfg)
print(f'training and testing took {(time.time() - started) / 60:.1f} min')

## 7. How training went

In [ ]:
history = json.load(open(os.path.join(OUT, 'history.json')))
loss = pd.DataFrame(history['loss'])
val = pd.DataFrame(history['val'])
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 4))
a.plot(loss['step'], loss['loss'], color='#2f70bb', lw=1.5)
a.set_title('training loss')
a.set_xlabel('step')
b.plot(val['step'], val['iou'], 'o-', color='#2f70bb', lw=2, label='walls')
b.plot(val['step'], val['openings_iou'], 'o-', color='#c98500', lw=2, label='doors and windows')
b.set_title(f'IoU on the {int(val["plans"].iloc[-1])} validation plans')
b.set_xlabel('step')
b.legend(frameon=False)
for ax in (a, b):
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
plt.show()
best = val.loc[val['iou'].idxmax()]
print(f'{int(loss["step"].iloc[-1])} steps in {loss["minutes"].iloc[-1]:.0f} min at about {loss["crops_per_s"].median():.0f} crops/s; '
      f'best validation wall IoU {best["iou"]:.3f} at step {int(best["step"])}')

## 8. The test plans

The kept checkpoint against the hand-written rules that FloorPlan3D's *Image to plan* uses today. The rules' scores below were
measured by the app's `scripts/evaluate-cubicasa.ts` on the same 400 test plans with the same truth and the same IoU; this
notebook's scorer draws the identical truth, so the two columns compare like with like.

In [ ]:
# Wall IoU of the rule pipeline (OpenCV.js, "Fill outlined walls" 9 px) on each test plan, from the FloorPlan3D repository.
RULES_IOU = {
    'high_quality_architectural/1191': 0.3347, 'high_quality_architectural/2536': 0.2595, 'high_quality_architectural/2530': 0.4436,
    'high_quality_architectural/2504': 0.4322, 'high_quality_architectural/2207': 0.1946, 'high_quality_architectural/22': 0.221,
    'high_quality_architectural/2090': 0.3608, 'high_quality_architectural/2085': 0.2128, 'high_quality_architectural/2014': 0.3983,
    'high_quality_architectural/2011': 0.4248, 'high_quality_architectural/1982': 0.4156, 'high_quality_architectural/1944': 0.2917,
    'high_quality_architectural/2539': 0.3024, 'high_quality_architectural/123': 0.4905, 'high_quality_architectural/6457': 0.3486,
    'high_quality_architectural/4812': 0.1116, 'high_quality_architectural/1754': 0.3586, 'high_quality_architectural/6066': 0.2029,
    'high_quality_architectural/4845': 0.7212, 'high_quality_architectural/875': 0.479, 'high_quality_architectural/7974': 0.3636,
    'high_quality_architectural/9488': 0.3864, 'high_quality_architectural/9093': 0.4454, 'high_quality_architectural/8269': 0.332,
    'high_quality_architectural/6187': 0.6257, 'high_quality_architectural/900': 0.5498, 'high_quality_architectural/5642': 0.3394,
    'high_quality_architectural/6378': 0.4794, 'high_quality_architectural/8138': 0.6228, 'high_quality_architectural/1235': 0.3371,
    'high_quality_architectural/4014': 0.2077, 'high_quality_architectural/366': 0.4097, 'high_quality_architectural/3802': 0.5235,
    'high_quality_architectural/1701': 0.4147, 'high_quality_architectural/6483': 0.3375, 'high_quality_architectural/207': 0.3231,
    'high_quality_architectural/5023': 0.3248, 'high_quality_architectural/9185': 0.192, 'high_quality_architectural/6704': 0.5133,
    'high_quality_architectural/6094': 0.2233, 'high_quality_architectural/84': 0.2153, 'high_quality_architectural/2031': 0.2691,
    'high_quality_architectural/32': 0.4185, 'high_quality_architectural/9223': 0.3193, 'high_quality_architectural/3038': 0.3962,
    'high_quality_architectural/1285': 0.3223, 'high_quality_architectural/1284': 0.2352, 'high_quality_architectural/6269': 0.1874,
    'high_quality_architectural/7647': 0.3889, 'high_quality_architectural/4068': 0.4951, 'high_quality_architectural/435': 0.1227,
    'high_quality_architectural/6697': 0.3594, 'high_quality_architectural/5135': 0.1888, 'high_quality_architectural/9274': 0.5147,
    'high_quality_architectural/8372': 0.4408, 'high_quality_architectural/5570': 0.3743, 'high_quality_architectural/1339': 0.0935,
    'high_quality_architectural/4847': 0.5438, 'high_quality_architectural/1293': 0.3437, 'high_quality_architectural/2510': 0.3187,
    'high_quality_architectural/1444': 0.0576, 'high_quality_architectural/1551': 0.149, 'high_quality_architectural/1590': 0.4933,
    'high_quality_architectural/20028': 0.5595, 'high_quality_architectural/1599': 0.4747, 'high_quality_architectural/1601': 0.1764,
    'high_quality_architectural/1636': 0.567, 'high_quality_architectural/165': 0.4815, 'high_quality_architectural/1669': 0.4106,
    'high_quality_architectural/7760': 0.4934, 'high_quality_architectural/52': 0.3599, 'high_quality_architectural/2035': 0.4563,
    'high_quality_architectural/521': 0.4812, 'high_quality_architectural/6596': 0.4497, 'high_quality_architectural/9976': 0.3633,
    'high_quality_architectural/9727': 0.2657, 'high_quality_architectural/3517': 0.3546, 'high_quality_architectural/6152': 0.4755,
    'high_quality_architectural/479': 0.3543, 'high_quality_architectural/7696': 0.7879, 'high_quality_architectural/3527': 0.1882,
    'high_quality_architectural/6166': 0.4558, 'high_quality_architectural/1683': 0.159, 'high_quality_architectural/6123': 0.4911,
    'high_quality_architectural/9228': 0.2367, 'high_quality_architectural/5956': 0.1539, 'high_quality_architectural/172': 0.5969,
    'high_quality_architectural/7858': 0.2852, 'high_quality_architectural/7809': 0.4273, 'high_quality_architectural/1845': 0.276,
    'high_quality_architectural/8726': 0.3915, 'high_quality_architectural/7894': 0.4558, 'high_quality_architectural/1242': 0.4552,
    'high_quality_architectural/9880': 0.2092, 'high_quality_architectural/2053': 0.3483, 'high_quality_architectural/9489': 0.2607,
    'high_quality_architectural/8189': 0.4508, 'high_quality_architectural/8281': 0.5217, 'high_quality_architectural/6467': 0.2181,
    'high_quality_architectural/9422': 0.2595, 'high_quality_architectural/239': 0.3504, 'high_quality_architectural/4928': 0.5194,
    'high_quality_architectural/1788': 0.4514, 'high_quality_architectural/6452': 0.2124, 'high_quality_architectural/141': 0.5114,
    'high_quality_architectural/6053': 0.3499, 'high_quality_architectural/3984': 0.4462, 'high_quality_architectural/6428': 0.4846,
    'high_quality_architectural/3781': 0.3041, 'high_quality_architectural/774': 0.3895, 'high_quality_architectural/6017': 0.1135,
    'high_quality_architectural/145': 0.3825, 'high_quality_architectural/6606': 0.3258, 'high_quality_architectural/3580': 0.3913,
    'high_quality_architectural/9886': 0.1673, 'high_quality_architectural/4594': 0.3057, 'high_quality_architectural/5018': 0.3663,
    'high_quality_architectural/7980': 0.4242, 'high_quality_architectural/4911': 0.5111, 'high_quality_architectural/1767': 0.3893,
    'high_quality_architectural/1966': 0.4364, 'high_quality_architectural/4553': 0.4303, 'high_quality_architectural/789': 0.4112,
    'high_quality_architectural/8304': 0.4898, 'high_quality_architectural/6259': 0.3124, 'high_quality_architectural/9453': 0.3167,
    'high_quality_architectural/4707': 0.2968, 'high_quality_architectural/392': 0.1907, 'high_quality_architectural/4535': 0.2177,
    'high_quality_architectural/1778': 0.5224, 'high_quality_architectural/481': 0.3562, 'high_quality_architectural/1652': 0.3336,
    'high_quality_architectural/6316': 0.3723, 'high_quality_architectural/6726': 0.5343, 'high_quality_architectural/8208': 0.4991,
    'high_quality_architectural/6549': 0.4257, 'high_quality_architectural/9577': 0.511, 'high_quality_architectural/1814': 0.4011,
    'high_quality_architectural/7639': 0.252, 'high_quality_architectural/6144': 0.4611, 'high_quality_architectural/1856': 0.4075,
    'high_quality_architectural/9452': 0.3713, 'high_quality_architectural/1869': 0.567, 'high_quality_architectural/6716': 0.4545,
    'high_quality_architectural/9692': 0.2205, 'high_quality_architectural/5121': 0.5031, 'high_quality_architectural/8576': 0.4936,
    'high_quality_architectural/5089': 0.1206, 'high_quality_architectural/225': 0.2177, 'high_quality_architectural/9198': 0.2621,
    'high_quality_architectural/6276': 0.1499, 'high_quality_architectural/6081': 0.3625, 'high_quality_architectural/5765': 0.3714,
    'high_quality_architectural/2149': 0.3543, 'high_quality_architectural/1837': 0.4898, 'high_quality_architectural/287': 0.2239,
    'high_quality_architectural/416': 0.4642, 'high_quality_architectural/6355': 0.3287, 'high_quality_architectural/654': 0.5085,
    'high_quality_architectural/1041': 0.5838, 'high_quality_architectural/1772': 0.4047, 'high_quality_architectural/1165': 0.3865,
    'high_quality_architectural/812': 0.4764, 'high_quality_architectural/9543': 0.236, 'high_quality_architectural/199': 0.3822,
    'high_quality_architectural/6408': 0.5068, 'high_quality_architectural/5064': 0.4426, 'high_quality_architectural/1801': 0.227,
    'high_quality_architectural/1838': 0.5942, 'high_quality_architectural/5617': 0.1815, 'high_quality_architectural/7037': 0.4038,
    'high_quality_architectural/4898': 0.43, 'high_quality_architectural/1228': 0.3132, 'high_quality_architectural/1839': 0.4139,
    'high_quality_architectural/1781': 0.1489, 'high_quality_architectural/9785': 0.064, 'high_quality_architectural/9540': 0.4285,
    'high_quality_architectural/6064': 0.2481, 'high_quality_architectural/5156': 0.2874, 'high_quality_architectural/9224': 0.4886,
    'high_quality_architectural/1914': 0.6799, 'high_quality_architectural/3608': 0.426, 'high_quality_architectural/9598': 0.4935,
    'high_quality_architectural/6059': 0.4372, 'high_quality_architectural/4813': 0.2855, 'high_quality_architectural/2029': 0.3973,
    'high_quality_architectural/1622': 0.2598, 'high_quality_architectural/1217': 0.4615, 'high_quality_architectural/1264': 0.5506,
    'high_quality_architectural/1849': 0.3833, 'high_quality_architectural/1116': 0.4139, 'high_quality_architectural/7780': 0.4645,
    'high_quality_architectural/7119': 0.2947, 'high_quality_architectural/8573': 0.5342, 'high_quality_architectural/211': 0.3218,
    'high_quality_architectural/9572': 0.3654, 'high_quality_architectural/1319': 0.3149, 'high_quality_architectural/5872': 0.5892,
    'high_quality_architectural/2582': 0.4758, 'high_quality_architectural/1945': 0.385, 'high_quality_architectural/1279': 0.4901,
    'high_quality_architectural/8571': 0.4877, 'high_quality_architectural/6007': 0.187, 'high_quality_architectural/5760': 0.5228,
    'high_quality_architectural/3890': 0.3312, 'high_quality_architectural/8285': 0.3834, 'high_quality_architectural/3860': 0.299,
    'high_quality_architectural/8046': 0.2698, 'high_quality_architectural/6042': 0.4863, 'high_quality_architectural/2036': 0.1211,
    'high_quality_architectural/6402': 0.3435, 'high_quality_architectural/5571': 0.2758, 'high_quality_architectural/5974': 0.0863,
    'high_quality_architectural/8502': 0.4508, 'high_quality_architectural/7071': 0.1425, 'high_quality_architectural/2037': 0.1811,
    'high_quality_architectural/5886': 0.0737, 'high_quality_architectural/9531': 0.4579, 'high_quality_architectural/7705': 0.5027,
    'high_quality_architectural/7901': 0.5128, 'high_quality_architectural/1841': 0.5187, 'high_quality_architectural/9443': 0.3926,
    'high_quality_architectural/5046': 0.3431, 'high_quality_architectural/1842': 0.3981, 'high_quality_architectural/7516': 0.1566,
    'high_quality_architectural/546': 0.2301, 'high_quality_architectural/552': 0.2191, 'high_quality_architectural/6631': 0.5165,
    'high_quality_architectural/4828': 0.4337, 'high_quality_architectural/6652': 0.2896, 'high_quality_architectural/1511': 0.2543,
    'high_quality_architectural/3868': 0.5508, 'high_quality_architectural/1910': 0.425, 'high_quality_architectural/3060': 0.4003,
    'high_quality_architectural/9796': 0.4675, 'high_quality_architectural/3660': 0.4588, 'high_quality_architectural/7606': 0.4096,
    'high_quality_architectural/5100': 0.2679, 'high_quality_architectural/7061': 0.3802, 'high_quality_architectural/5039': 0.4328,
    'high_quality_architectural/4555': 0.5707, 'high_quality_architectural/541': 0.6704, 'high_quality_architectural/2204': 0.3137,
    'high_quality_architectural/4589': 0.3326, 'high_quality_architectural/20000': 0.2811, 'high_quality_architectural/20002': 0.1799,
    'high_quality_architectural/20003': 0.4531, 'high_quality_architectural/20017': 0.5347, 'high_quality_architectural/20019': 0.2526,
    'high_quality_architectural/20021': 0.5669, 'high_quality_architectural/20022': 0.3434, 'high_quality_architectural/1921': 0.2421,
    'high_quality_architectural/4775': 0.5566, 'high_quality_architectural/3516': 0.3037, 'high_quality_architectural/6023': 0.5058,
    'high_quality_architectural/9433': 0.5152, 'high_quality_architectural/591': 0.5932, 'colorful/11260': 0.3078,
    'colorful/11395': 0.5493, 'colorful/11397': 0.5089, 'colorful/11398': 0.4984,
    'colorful/11399': 0.5258, 'colorful/11400': 0.4948, 'colorful/11402': 0.5441,
    'colorful/11485': 0.6786, 'colorful/11527': 0.4903, 'colorful/11565': 0.6621,
    'colorful/11573': 0.6071, 'colorful/11578': 0.4271, 'colorful/11664': 0.6085,
    'colorful/11750': 0.5758, 'colorful/11752': 0.5382, 'colorful/11773': 0.6401,
    'colorful/11774': 0.6474, 'colorful/11859': 0.5309, 'colorful/11867': 0.532,
    'colorful/11960': 0.7281, 'colorful/11962': 0.5194, 'colorful/12002': 0.6367,
    'colorful/12030': 0.2287, 'colorful/12094': 0.6843, 'colorful/12139': 0.5857,
    'colorful/12200': 0.6639, 'colorful/12202': 0.7304, 'colorful/12214': 0.4858,
    'colorful/12217': 0.5326, 'colorful/12229': 0.5613, 'colorful/12560': 0.6663,
    'colorful/12625': 0.4585, 'colorful/12627': 0.6013, 'colorful/12729': 0.589,
    'colorful/12769': 0.5573, 'colorful/12787': 0.2594, 'colorful/12830': 0.4051,
    'colorful/12866': 0.4489, 'colorful/12871': 0.5798, 'colorful/12872': 0.0898,
    'colorful/12946': 0.5766, 'colorful/12981': 0.6758, 'colorful/13019': 0.4068,
    'colorful/13022': 0.4049, 'colorful/13023': 0.5104, 'colorful/13031': 0.4448,
    'colorful/13138': 0.644, 'colorful/13181': 0.4763, 'colorful/13195': 0.3442,
    'colorful/13213': 0.4359, 'colorful/13226': 0.3248, 'colorful/13254': 0.6094,
    'colorful/13258': 0.5231, 'colorful/13264': 0.4322, 'colorful/13278': 0.6478,
    'colorful/13281': 0.4286, 'colorful/13352': 0.4111, 'colorful/13438': 0.4576,
    'colorful/13500': 0.4941, 'colorful/13502': 0.5415, 'colorful/13577': 0.5257,
    'colorful/13589': 0.574, 'colorful/13685': 0.5629, 'colorful/2586': 0.3539,
    'colorful/9448': 0.6889, 'high_quality/1140': 0.2963, 'high_quality/3512': 0.5484,
    'high_quality/1569': 0.8446, 'high_quality/9864': 0.6865, 'high_quality/9803': 0.3866,
    'high_quality/4712': 0.6469, 'high_quality/9480': 0.6297, 'high_quality/5774': 0.5162,
    'high_quality/9334': 0.4607, 'high_quality/6058': 0.5278, 'high_quality/8380': 0.7176,
    'high_quality/5703': 0.4868, 'high_quality/5836': 0.5382, 'high_quality/8731': 0.653,
    'high_quality/4815': 0.5787, 'high_quality/3930': 0.4671, 'high_quality/3106': 0.6388,
    'high_quality/8816': 0.332, 'high_quality/5588': 0.6744, 'high_quality/3162': 0.6532,
    'high_quality/3638': 0.7906, 'high_quality/5807': 0.6293, 'high_quality/90': 0.5183,
    'high_quality/8230': 0.7569, 'high_quality/3801': 0.5468, 'high_quality/4767': 0.125,
    'high_quality/7625': 0.591, 'high_quality/3676': 0.6685, 'high_quality/1025': 0.4871,
    'high_quality/5858': 0.7508, 'high_quality/1692': 0.5774, 'high_quality/1506': 0.713,
    'high_quality/337': 0.5067, 'high_quality/3078': 0.3238, 'high_quality/1282': 0.2832,
    'high_quality/1877': 0.6092, 'high_quality/3529': 0.7142, 'high_quality/750': 0.6726,
    'high_quality/4851': 0.6517, 'high_quality/837': 0.5854, 'high_quality/477': 0.538,
    'high_quality/9671': 0.6582, 'high_quality/3074': 0.2274, 'high_quality/3141': 0.625,
    'high_quality/5985': 0.4717, 'high_quality/1403': 0.6809, 'high_quality/3247': 0.5007,
    'high_quality/7790': 0.7957, 'high_quality/19': 0.4838, 'high_quality/3954': 0.5479,
    'high_quality/6624': 0.5128, 'high_quality/6521': 0.5256, 'high_quality/5632': 0.5406,
    'high_quality/488': 0.4372, 'high_quality/4617': 0.6093, 'high_quality/1510': 0.5773,
    'high_quality/5845': 0.6304, 'high_quality/20174': 0.5889, 'high_quality/20183': 0.6813,
    'high_quality/20188': 0.6091, 'high_quality/20197': 0.7333, 'high_quality/20198': 0.6144,
    'high_quality_architectural/7691': 0.4831, 'high_quality_architectural/1432': 0.4395, 'high_quality_architectural/1718': 0.4146,
    'high_quality_architectural/1141': 0.5089, 'high_quality_architectural/9270': 0.5303, 'high_quality_architectural/772': 0.6724,
    'high_quality_architectural/6303': 0.364, 'colorful/3598': 0.7853, 'colorful/8389': 0.6478,
    'high_quality_architectural/20023': 0.2494, 'high_quality_architectural/2179': 0.4059, 'high_quality_architectural/7715': 0.0494,
    'high_quality_architectural/9384': 0.3645, 'high_quality_architectural/3024': 0.5783, 'high_quality_architectural/18': 0.1702,
    'high_quality/20168': 0.6074,
}

In [ ]:
results = json.load(open(os.path.join(OUT, 'test_results.json')))
rows = pd.DataFrame(results['rows']).set_index('plan')
rows['kind'] = [kind(p) for p in rows.index]
rows['rules_iou'] = [RULES_IOU.get(p, np.nan) for p in rows.index]
s = results['summary']
both = rows.dropna(subset=['rules_iou'])
display(Markdown(
    f"| on {s['plans']} test plans | trained model | rules |\n|---|---|---|\n"
    f"| wall IoU (mean) | **{s['iou']:.1%}** | {both['rules_iou'].mean():.1%} |\n"
    f"| wall IoU (median) | {s['median_iou']:.1%} | {both['rules_iou'].median():.1%} |\n"
    f"| precision / recall | {s['precision']:.1%} / {s['recall']:.1%} | |\n"
    f"| door and window IoU | {s['openings_iou']:.1%} | not found by the rules |\n"
    f"| model better on | {(both['iou'] > both['rules_iou']).sum()} of {len(both)} plans | |\n"
    f"| checkpoint | step {s['best_step']} | |"
))
display(rows.groupby('kind')[['iou', 'rules_iou', 'openings_iou']].mean().round(3).rename(
    columns={'iou': 'model wall IoU', 'rules_iou': 'rules wall IoU', 'openings_iou': 'model door+window IoU'}))

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(15, 5.5))
a.scatter(both['rules_iou'], both['iou'], s=14, color='#2f70bb', alpha=0.6)
a.plot([0, 1], [0, 1], color='grey', lw=1)
a.set_xlim(0, 1)
a.set_ylim(0, 1)
a.set_xlabel('rules, wall IoU')
a.set_ylabel('trained model, wall IoU')
a.set_title('each dot is one test plan; above the line the model is better')
bins = np.linspace(0, 1, 26)
b.hist(both['rules_iou'], bins=bins, color='#c98500', alpha=0.8, label='rules')
b.hist(both['iou'], bins=bins, color='#2f70bb', alpha=0.8, label='trained model')
b.set_xlabel('wall IoU')
b.set_ylabel('plans')
b.legend(frameon=False)
for ax in (a, b):
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
plt.show()

## 9. Where it goes wrong

The best, a typical (median) and the worst test plan. Teal is wall found, red is wall found where there is none, amber is wall missed.

In [ ]:
def errors(gray: np.ndarray, labels: np.ndarray, pred: np.ndarray) -> np.ndarray:
    truth, found = labels == 1, pred == 1
    out = np.repeat((165 + gray[..., None] * 0.35).astype(np.uint8), 3, axis=2)
    out[truth & found] = (27, 108, 128)
    out[found & ~truth] = (229, 86, 111)
    out[truth & ~found] = (232, 160, 32)
    return out


ranked = rows.sort_values('iou')
picks = {'best': ranked.index[-1], 'typical': ranked.index[len(ranked) // 2], 'worst': ranked.index[0]}
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (label, plan) in zip(axes, picks.items()):
    gray, labels = load(plan)
    pred = cv2.imread(os.path.join(OUT, 'pred_test', plan_name(plan) + '.png'), cv2.IMREAD_UNCHANGED)
    ax.imshow(around((labels > 0) | (pred > 0), errors(gray, labels, pred))[0])
    ax.set_title(f'{label}: {plan}, wall IoU {rows.loc[plan, "iou"]:.1%} (rules {rows.loc[plan, "rules_iou"]:.1%})')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Export for the browser

FloorPlan3D runs in the browser, so the model is exported to ONNX for onnxruntime-web, with the image height and width left free.
The export is checked against PyTorch on a tile of a test plan. `model_card.json` records what the browser needs to feed it.

In [ ]:
checkpoint = torch.load(os.path.join(OUT, 'model_best.pt'), map_location='cpu')
export_net = UNet(**checkpoint['config']).eval()
export_net.load_state_dict(checkpoint['state'])
onnx_path = os.path.join(OUT, 'floorplan3d_walls.onnx')
dummy = torch.zeros(1, 1, 512, 512)
exported = False
for dynamo in (False, True):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', DeprecationWarning)  # the TorchScript exporter is deprecated but still the simplest here
            torch.onnx.export(export_net, (dummy,), onnx_path, input_names=['ink'], output_names=['logits'], opset_version=17,
                              dynamic_axes={'ink': {2: 'height', 3: 'width'}, 'logits': {2: 'height', 3: 'width'}}, dynamo=dynamo)
        exported = True
        print(f'exported {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB, {"dynamo" if dynamo else "TorchScript"} exporter)')
        break
    except Exception as err:  # the exporters need optional packages; report and try the other one
        print(f'{"dynamo" if dynamo else "TorchScript"} exporter failed: {type(err).__name__}: {str(err)[:200]}')

if exported:
    try:
        import onnxruntime as ort
        gray, _ = load(splits['test'][0])
        tile = np.full((1024, 1024), 255, np.uint8)
        h, w = min(1024, gray.shape[0]), min(1024, gray.shape[1])
        tile[:h, :w] = gray[:h, :w]
        x = ink(tile)[None, None]
        onnx_classes = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider']).run(None, {'ink': x})[0].argmax(1)
        with torch.no_grad():
            torch_classes = export_net(torch.from_numpy(x)).argmax(1).numpy()
        print(f'ONNX and PyTorch agree on {(onnx_classes == torch_classes).mean():.4%} of the pixels of a 1,024 px test tile')
    except ImportError:
        print('onnxruntime is not installed here; check the export with training/check_onnx.py on a machine that has it')

card = {
    'model': 'FloorPlan3D wall model (U-Net)', 'classes': list(CLASSES), 'config': checkpoint['config'], 'checkpoint_step': checkpoint['step'],
    'input': 'ink = 1 - gray / 255 (0 paper, 1 black), shape [1, 1, H, W] float32, H and W multiples of 32',
    'tiling': {'tile': 1024, 'margin': 96}, 'onnx': os.path.basename(onnx_path) if exported else None,
    'validation': checkpoint['val'], 'test': results['summary'],
    'data': 'CubiCasa5k (CC BY-NC-SA 4.0): non-commercial use only',
}
json.dump(card, open(os.path.join(OUT, 'model_card.json'), 'w'), indent=1)

## 11. What this run leaves

`model_best.pt` (the kept checkpoint), `model_last.pt`, `history.json` and `log.txt` (the training record), `test_results.json` and
`pred_test/` (scores and class maps of the test plans), `floorplan3d_walls.onnx` and `model_card.json` (for the browser).

In [ ]:
for name in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, name)
    size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) if os.path.isdir(path) else os.path.getsize(path)
    print(f'{name:28s} {size / 1e6:8.1f} MB')